In [2]:
import networkx as nx
import scipy as sp
import numpy as np
import pandas as pd
import graph_tool.all as gt
import gc

l = pd.read_csv("4932.protein.links.v12.0.txt",sep=r"\s+")

#links stores the pairwise edge score (0-1000) <-- remove header from file
full_links = l[l["combined_score"] >= 750].copy()

print(full_links.shape)
print(full_links.columns)
print(full_links.memory_usage(deep=True).sum() / 1e9, "GB")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

(183006, 3)
Index(['protein1', 'protein2', 'combined_score'], dtype='str')
0.028199928 GB


In [7]:
# 1. Basic cleaning

links = full_links[["protein1", "protein2", "combined_score"]].dropna().copy()
links["combined_score"] = links["combined_score"].astype(np.float32)

if links["combined_score"].max() > 1:
    links["combined_score"] /= 1000.0

assert links["combined_score"].between(0, 1).all()

print("Rows before cleaning:", len(links))

# Remove self-loops
links = links[links["protein1"] != links["protein2"]].copy()

# 2. Canonicalise full network

pairs = np.sort(links[["protein1", "protein2"]].astype(str).to_numpy(),axis=1)

clean_full_links = pd.DataFrame({
    "protein1": pairs[:, 0],
    "protein2": pairs[:, 1],
    "combined_score": links["combined_score"].to_numpy()})

del pairs
gc.collect()

clean_full_links = (
    clean_full_links
    .groupby(
        ["protein1", "protein2"],
        as_index=False,
        sort=False
    )["combined_score"]
    .max()
)

assert not clean_full_links.duplicated(
    ["protein1", "protein2"]
).any()


# 3. Build full graph used only for distances
full_proteins = pd.Index(
    pd.unique(
        pd.concat(
            [
                clean_full_links["protein1"],
                clean_full_links["protein2"]
            ],
            ignore_index=True
        )
    )
)

full_protein_to_vertex = pd.Series(
    np.arange(len(full_proteins), dtype=np.int32),
    index=full_proteins
)

full_vertex_to_protein = dict(
    enumerate(full_proteins)
)

full_src = (
    clean_full_links["protein1"]
    .map(full_protein_to_vertex)
    .to_numpy(dtype=np.int32)
)

full_dst = (
    clean_full_links["protein2"]
    .map(full_protein_to_vertex)
    .to_numpy(dtype=np.int32)
)

g_full = gt.Graph(directed=False)

g_full.add_edge_list(
    np.column_stack((full_src, full_dst))
)

print("\nFULL GRAPH")
print("Vertices:", g_full.num_vertices())
print("Edges:", g_full.num_edges())

# 4. Calculate distances from HSC82 on the full graph
target_protein = "4932.YMR186W"

target_idx_full = int(full_protein_to_vertex[target_protein])

distances_full = gt.shortest_distance(g_full,source=g_full.vertex(target_idx_full))

distance_table = pd.DataFrame({
    "protein": [
        full_vertex_to_protein[i]
        for i in range(g_full.num_vertices())
    ],
    "distance_from_target": [
        int(distances_full[g_full.vertex(i)])
        for i in range(g_full.num_vertices())
    ]
})

distance_table = distance_table[
    distance_table["protein"] != target_protein
].copy()

proteins = full_proteins
clean_links = clean_full_links

protein_to_vertex = pd.Series(
    np.arange(len(proteins), dtype=np.int32),
    index=proteins
)

vertex_to_protein = dict(
    enumerate(proteins)
)

src = (
    clean_links["protein1"]
    .map(protein_to_vertex)
    .to_numpy(dtype=np.int32)
)

dst = (
    clean_links["protein2"]
    .map(protein_to_vertex)
    .to_numpy(dtype=np.int32)
)

q_values = (
    clean_links["combined_score"]
    .to_numpy(dtype=np.float64)
)

g_obs = gt.Graph(directed=False)

g_obs.add_edge_list(
    np.column_stack((src, dst))
)

q_obs = g_obs.new_ep("double")
q_obs.a = q_values
g_obs.ep["q"] = q_obs

assert g_obs.num_edges() == len(clean_links)

print("GRAPH FOR uSBM")
print("Vertices:", g_obs.num_vertices())
print("Edges:", g_obs.num_edges())

# 8. Finds proteins that satisfies the distance requirement in g_obs 

distant_proteins = distance_table[
    (distance_table["distance_from_target"] >= 3)
    &
    (distance_table["protein"].isin(protein_to_vertex.index))
].copy()

#Remove missing values
links = full_links[["protein1", "protein2", "combined_score"]].dropna().copy()

#Convert STRING scores from 0-1000 to 0-1
links["combined_score"] = links["combined_score"].astype(np.float32)
if links["combined_score"].max() > 1:
    links["combined_score"] /= 1000.0
assert links["combined_score"].between(0, 1).all()

# Remove self-loops
links = links[links["protein1"] != links["protein2"]].copy()

#Canonicalise each undirected pair, i.e. A-B and B-A both become A-B
pairs = np.sort(links[["protein1", "protein2"]].astype(str).to_numpy(),axis=1)
clean_links = pd.DataFrame({"protein1": pairs[:, 0], "protein2": pairs[:, 1], "combined_score": links["combined_score"].to_numpy()})
del pairs
gc.collect()

#Remove reverse duplicates
clean_links = (clean_links.groupby(["protein1", "protein2"],as_index=False,sort=False)["combined_score"].max())
print("Unique undirected edges:", len(clean_links))
assert not clean_links.duplicated(["protein1", "protein2"]).any()
print(len(clean_links))

#Create vertex mapping
proteins = pd.Index(pd.unique(pd.concat([clean_links["protein1"],clean_links["protein2"]],ignore_index=True)))
protein_to_vertex = pd.Series(np.arange(len(proteins), dtype=np.int32),index=proteins)
vertex_to_protein = dict(enumerate(proteins))

# Convert edges to integer vertex indices
src = clean_links["protein1"].map(protein_to_vertex).to_numpy(dtype=np.int32)
dst = clean_links["protein2"].map(protein_to_vertex).to_numpy(dtype=np.int32)
q_values = clean_links["combined_score"].to_numpy(dtype=np.float64)

# Build graph-tool graph
g_obs = gt.Graph(directed=False)
edge_array = np.column_stack((src, dst))
g_obs.add_edge_list(edge_array)
q_obs = g_obs.new_ep("double")
q_obs.a = q_values
g_obs.ep["q"] = q_obs



Rows before cleaning: 183006

FULL GRAPH
Vertices: 5682
Edges: 91503
GRAPH FOR uSBM
Vertices: 5682
Edges: 91503
Unique undirected edges: 91503
91503


In [3]:
#Fitting initial nested SBM
initial_bstate = gt.minimize_nested_blockmodel_dl(g_obs)
print("Number of hierarchy levels:", len(initial_bstate.get_bs()))

#Creating uncertain SBM
initial_bs = initial_bstate.get_bs()

state = gt.UncertainBlockState(
    g_obs,
    q=q_obs,
    q_default=0.0,
    nested=True,
    state_args={"bs": initial_bs})

print(type(state))
print(type(state.get_block_state()))

print("Vertices:", g_obs.num_vertices())
print("Observed edges:", g_obs.num_edges())
print("Initial entropy:", state.entropy())

Number of hierarchy levels: 14
<class 'graph_tool.inference.uncertain_blockmodel.UncertainBlockState'>
<class 'graph_tool.inference.nested_blockmodel.NestedBlockState'>
Vertices: 5682
Observed edges: 91503
Initial entropy: -316283.79412946606


In [4]:
gt.mcmc_equilibrate(
    state,
    wait=30,
    nbreaks=2,
    epsilon=0.001,
    mcmc_args={
        "beta": 1,
        "niter": 1,
        "pedges": 0.01,
        "multiflip": False
    },
    verbose=True
)

niter:     1  count:    0  breaks:  0  min_S: -316283.79  max_S: -316156.19  S: -316156.19  ΔS:      127.603  moves:    71 
niter:     2  count:    0  breaks:  0  min_S: -316283.79  max_S: -316037.13  S: -316037.13  ΔS:      119.057  moves:    83 
niter:     3  count:    0  breaks:  0  min_S: -316283.79  max_S: -315918.60  S: -315918.60  ΔS:      118.530  moves:    98 
niter:     4  count:    0  breaks:  0  min_S: -316283.79  max_S: -315888.66  S: -315888.66  ΔS:      29.9453  moves:    97 
niter:     5  count:    0  breaks:  0  min_S: -316283.79  max_S: -315776.52  S: -315776.52  ΔS:      112.142  moves:    99 
niter:     6  count:    0  breaks:  0  min_S: -316283.79  max_S: -315739.41  S: -315739.41  ΔS:      37.1067  moves:   103 
niter:     7  count:    1  breaks:  0  min_S: -316283.79  max_S: -315739.41  S: -315769.69  ΔS:     -30.2809  moves:   101 
niter:     8  count:    2  breaks:  0  min_S: -316283.79  max_S: -315739.41  S: -315792.69  ΔS:     -22.9951  moves:   114 
niter:  

(-315917.8765853118, 1004472, 19583)

In [5]:
#Collect posterior partitions
import pickle

partition_samples = []

for i in range(100):

    # Advance MCMC by 10 updates
    for _ in range(10):
        state.mcmc_sweep(
            niter=1,
            pedges=0.01,
            multiflip=False
        )

    # Save current nested partition
    bs = state.get_block_state().get_bs()

    partition_samples.append([
        np.asarray(b).copy() for b in bs
    ])

    # Save every 10 samples in case the kernel crashes
    if (i + 1) % 10 == 0:
        with open("partition_samples.pkl", "wb") as f:
            pickle.dump(partition_samples, f)

        print(f"Collected {i+1} samples", flush=True)

#Consensus Partition
pmode = gt.PartitionModeState(
    partition_samples,
    nested=True,
    converge=True
)

consensus_bs = pmode.get_max_nested()

# Finest-level consensus partition
consensus_labels = np.asarray(consensus_bs[0])

print("Number of blocks:", len(np.unique(consensus_labels)))

#Find HSC82 consensus block
hsc82_idx = int(protein_to_vertex["4932.YMR186W"])

hsc82_block = consensus_labels[hsc82_idx]

members = np.where(consensus_labels == hsc82_block)[0]

print("HSC82 block:", hsc82_block)
print("Block size:", len(members))

for idx in members:
    print(vertex_to_protein[int(idx)])

Collected 10 samples
Collected 20 samples
Collected 30 samples
Collected 40 samples
Collected 50 samples
Collected 60 samples
Collected 70 samples
Collected 80 samples
Collected 90 samples
Collected 100 samples
Number of blocks: 341
HSC82 block: 336
Block size: 2
4932.YMR186W
4932.YPL240C


In [6]:
# Outputs partitions for each hierarchy level
target_protein = "4932.YMR186W"  # HSC82
target_idx = int(protein_to_vertex[target_protein])

labels = np.asarray(consensus_bs[0]).copy()

results = []

for level in range(len(consensus_bs)):

    if level > 0:
        labels = np.asarray(consensus_bs[level])[labels]

    # Number of proteins in each block
    unique_blocks, block_sizes = np.unique(
        labels, return_counts=True
    )

    # Block containing HSC82
    target_block = labels[target_idx]
    target_block_size = np.sum(labels == target_block)

    results.append({
        "Level": level,
        "Number of blocks": len(unique_blocks),
        "Smallest block": np.min(block_sizes),
        "Largest block": np.max(block_sizes),
        "Mean block size": np.mean(block_sizes),
        "HSC82 block size": target_block_size
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

 Level  Number of blocks  Smallest block  Largest block  Mean block size  HSC82 block size
     0               341               1            135        16.662757                 2
     1                92               5            351        61.760870                87
     2                31               8            531       183.290323               278
     3                11              52           1452       516.545455               911
     4                 4             160           3912      1420.500000              3912
     5                 1            5682           5682      5682.000000              5682
     6                 1            5682           5682      5682.000000              5682
     7                 1            5682           5682      5682.000000              5682
     8                 1            5682           5682      5682.000000              5682
     9                 1            5682           5682      5682.000000              5682

In [7]:
#Fining adjacent blocks/communities
target_protein = "4932.YMR186W"  # HSC82
target_idx = int(protein_to_vertex[target_protein])

# Choose hierarchical level
level = 1

# Assign every protein to its community at this level
labels = np.asarray(consensus_bs[0]).copy()

for l in range(1, level + 1):
    labels = np.asarray(consensus_bs[l])[labels]

# Community containing HSC82
target_block = labels[target_idx]

# Extract edges from the observed STRING network
edges = g_obs.get_edges()
source = edges[:, 0]
target = edges[:, 1]

# Find edges crossing the boundary of HSC82's community
mask1 = (labels[source] == target_block) & (labels[target] != target_block)
mask2 = (labels[target] == target_block) & (labels[source] != target_block)

# Identify communities on the other side
neighbour_blocks = np.concatenate([
    labels[target[mask1]],
    labels[source[mask2]]
])

# Count edges connecting each adjacent community
blocks, counts = np.unique(
    neighbour_blocks,
    return_counts=True
)

# Community sizes
sizes = np.bincount(labels)

# Results
df = pd.DataFrame({
    "Adjacent block": blocks,
    "Block size": sizes[blocks],
    "Edges to HSC82 block": counts
})

df = df.sort_values(
    "Edges to HSC82 block",
    ascending=False
)

print("HSC82 block:", target_block)
print("HSC82 block size:", sizes[target_block])
print(df.to_string(index=False))

HSC82 block: 8
HSC82 block size: 87
 Adjacent block  Block size  Edges to HSC82 block
             85          38                   144
             64          14                    91
             23         132                    49
             18          55                    28
             68          41                    26
             62          56                    24
             36          79                    22
             27         101                    22
              6          96                    22
             13          94                    20
             22           6                    13
             76          67                    13
             67          30                    12
             29          26                    11
             39         116                    10
             63          64                    10
             53          64                     9
             60          40                     8
             4

In [8]:
# Load STRING protein information
protein_info = pd.read_csv(
    "4932.protein.info.v12.0.txt",
    sep="\t"
)

print(protein_info.columns.tolist())

# Find proteins annotated as dubious
dubious_df = protein_info[
    protein_info["annotation"].str.contains(
        "dubious",
        case=False,
        na=False
    )
].copy()

# Extract STRING IDs
dubious_ids = set(dubious_df["#string_protein_id"])

print("Number of dubious proteins:", len(dubious_ids))
print(dubious_df[
    ["#string_protein_id", "preferred_name", "annotation"]
].head(10).to_string(index=False))

dubious_ids = dubious_ids.intersection(
    protein_to_vertex.keys()
)

print("Dubious proteins in network:", len(dubious_ids))

['#string_protein_id', 'preferred_name', 'protein_size', 'annotation']
Number of dubious proteins: 692
#string_protein_id preferred_name                                                                                                                                                                                                                                      annotation
        4932.Q0010          Q0010                       Putative uncharacterized protein Q0010, mitochondrial; Dubious open reading frame; unlikely to encode a protein, based on available experimental and comparative sequence data; partially overlaps the dubious ORF Q0017.
        4932.Q0017          Q0017                       Putative uncharacterized protein Q0017, mitochondrial; Dubious open reading frame; unlikely to encode a protein, based on available experimental and comparative sequence data; partially overlaps the dubious ORF Q0010.
        4932.Q0032          Q0032                                          

In [9]:
level = 1 # Hierarchical level to analyse

target_protein = "4932.YMR186W"  # HSC82

important_proteins = {
    "4932.YMR186W": 10.0,  # HSC82
    "4932.YAL005C": 10.0,  # SSA1
    "4932.YPL240C": 10.0   # HSP82 (Hsp90)
}


# 2. get consensus community assignments

labels = np.asarray(consensus_bs[0]).copy()

for l in range(1, level + 1):
    labels = np.asarray(consensus_bs[l])[labels]

target_idx = int(protein_to_vertex[target_protein])
target_block = labels[target_idx]

print("HSC82 community:", target_block)

# 3. find adjacnt communities

edges = g_obs.get_edges()[:, :2]

u = edges[:, 0].astype(int)
v = edges[:, 1].astype(int)

mask1 = (
    (labels[u] == target_block) &
    (labels[v] != target_block)
)

mask2 = (
    (labels[v] == target_block) &
    (labels[u] != target_block)
)

adjacent_blocks = np.unique(
    np.concatenate([
        labels[v[mask1]],
        labels[u[mask2]]
    ])
)

print("Adjacent communities:", adjacent_blocks)
print("Number of adjacent communities:", len(adjacent_blocks))

# 4. define protein weights

beta = g_obs.new_vertex_property("double", val=1.0)

# Dubious proteins receive lower baseline importance
for protein in dubious_ids:
    if protein in protein_to_vertex:
        idx = int(protein_to_vertex[protein])
        beta[g_obs.vertex(idx)] = 0.1

# Important proteins receive higher baseline importance
for protein, weight in important_proteins.items():
    if protein in protein_to_vertex:
        idx = int(protein_to_vertex[protein])
        beta[g_obs.vertex(idx)] = weight


# 5. Calculate Katz centrality

results = []

for block in adjacent_blocks:

    # Induced subgraph of this community
    g_block = gt.GraphView(
        g_obs,
        vfilt=(labels == block),
        directed=False
    )

    vertices = g_block.get_vertices()

    if len(vertices) == 0:
        continue

    # Choose alpha to ensure convergence
    max_degree = max(g_block.get_out_degrees(vertices))
    alpha = 0.5 / max(1, max_degree)

    # Personalised Katz centrality
    katz = gt.katz(
        g_block,
        alpha=alpha,
        beta=beta,
        norm=False
    )

    # Record results
    for v_idx in vertices:
        protein = vertex_to_protein[int(v_idx)]

        results.append({
            "Protein": protein,
            "Community": int(block),
            "Community size": len(vertices),
            "Katz score": float(katz[g_block.vertex(int(v_idx))]),
            "Dubious": protein in dubious_ids
        })

# 6. rank proteins according to katz in each community

df = pd.DataFrame(results)

if not df.empty:
    df = df.sort_values(
        ["Community", "Katz score"],
        ascending=[True, False]
    )

    df["Rank"] = (
        df.groupby("Community").cumcount() + 1
    )

    top10 = df[df["Rank"] <= 10].copy()

    print("\nTop 10 proteins per adjacent community:")
    print(top10.to_string(index=False))



HSC82 community: 8
Adjacent communities: [ 0  2  5  6  7  9 10 11 12 13 15 17 18 19 20 21 22 23 24 26 27 28 29 30
 31 33 36 37 38 39 40 41 42 43 44 45 46 47 48 50 51 52 53 58 59 60 61 62
 63 64 67 68 69 72 73 75 76 79 80 85 86 88 91 92 94]
Number of adjacent communities: 65

Top 10 proteins per adjacent community:
       Protein  Community  Community size  Katz score  Dubious  Rank
  4932.YBR271W          0              51    1.623562    False     1
  4932.YIL064W          0              51    1.424824    False     2
  4932.YIL110W          0              51    1.405100    False     3
  4932.YIL096C          0              51    1.379973    False     4
  4932.YLR137W          0              51    1.360992    False     5
  4932.YPL208W          0              51    1.334769    False     6
  4932.YDR465C          0              51    1.286181    False     7
  4932.YBR261C          0              51    1.266089    False     8
  4932.YBR030W          0              51    1.252381    False 

In [10]:
target_idx = int(protein_to_vertex["4932.YMR186W"])

# Calculate shortest-path distances
distances = gt.shortest_distance(
    g_obs,
    source=g_obs.vertex(target_idx),
    directed=False
).a

# Start with all proteins ranked by Katz centrality
candidates = df.copy()

# Add shortest-path distances
candidates["Distance from HSC82"] = [
    int(distances[int(protein_to_vertex[p])])
    for p in candidates["Protein"]
]

# Exclude dubious proteins
candidates = candidates[
    candidates["Dubious"] == False
].copy()

# Rank the eligible proteins within each community
candidates = candidates.sort_values(
    ["Community", "Katz score"],
    ascending=[True, False]
)

candidates["Eligible Rank"] = (
    candidates.groupby("Community").cumcount() + 1
)

# Top 10 eligible proteins per community
eligible_top10 = candidates[
    candidates["Eligible Rank"] <= 10
].copy()

print(eligible_top10.to_string(index=False))

       Protein  Community  Community size  Katz score  Dubious  Rank  Distance from HSC82  Eligible Rank
  4932.YBR271W          0              51    1.623562    False     1                    2              1
  4932.YIL064W          0              51    1.424824    False     2                    2              2
  4932.YIL110W          0              51    1.405100    False     3                    3              3
  4932.YIL096C          0              51    1.379973    False     4                    3              4
  4932.YLR137W          0              51    1.360992    False     5                    3              5
  4932.YPL208W          0              51    1.334769    False     6                    3              6
  4932.YDR465C          0              51    1.286181    False     7                    3              7
  4932.YBR261C          0              51    1.266089    False     8                    3              8
  4932.YBR030W          0              51    1.252381  

In [12]:
# 1. GET TOP 3 ELIGIBLE PROTEINS PER COMMUNITY

pool = eligible_top10.copy()

# Exclude dubious proteins
pool = pool[pool["Dubious"] == False].copy()

pool = pool.sort_values(
    ["Community", "Katz score"],
    ascending=[True, False]
)

top3_all = pool.groupby("Community").head(3).copy()

# Only communities with at least 3 eligible proteins
counts = top3_all.groupby("Community").size()
valid_blocks = counts[counts >= 3].index

# 2. CALCULATE INTER-COMMUNITY DENSITY

target_idx = int(protein_to_vertex["4932.YMR186W"])
target_block = labels[target_idx]

edges = g_obs.get_edges()[:, :2]

u = edges[:, 0].astype(int)
v = edges[:, 1].astype(int)

mask1 = (
    (labels[u] == target_block) &
    (labels[v] != target_block)
)

mask2 = (
    (labels[v] == target_block) &
    (labels[u] != target_block)
)

neighbour_blocks = np.concatenate([
    labels[v[mask1]],
    labels[u[mask2]]
])

# Number of edges to each adjacent community
blocks, edge_counts = np.unique(
    neighbour_blocks,
    return_counts=True
)

# Size of every community
sizes = np.bincount(labels)
target_size = sizes[target_block]

community_df = pd.DataFrame({
    "Community": blocks,
    "Edges": edge_counts,
    "Community size": sizes[blocks]
})

# Inter-community edge density
community_df["Density"] = (
    community_df["Edges"] /
    (target_size * community_df["Community size"])
)

# Only communities with 3 eligible candidates
community_df = community_df[
    community_df["Community"].isin(valid_blocks)
].copy()

# Rank communities by density
community_df = community_df.sort_values(
    ["Density", "Edges"],
    ascending=[False, False]
)

# Select top 10 communities
selected_blocks = community_df.head(10)["Community"]

print("Top 10 communities:")
print(community_df.head(10).to_string(index=False))

# 3. SELECT TOP 3 PROTEINS PER COMMUNITY

top30 = top3_all[
    top3_all["Community"].isin(selected_blocks)
].copy()

top30 = top30.merge(
    community_df[["Community", "Density"]],
    on="Community",
    how="left"
)

top30 = top30.sort_values(
    ["Density", "Community", "Katz score"],
    ascending=[False, True, False]
)

print("\nSelected proteins:", len(top30))
print(top30.to_string(index=False))



Top 10 communities:
 Community  Edges  Community size  Density
        64     91              14 0.074713
        85    144              38 0.043557
        22     13               6 0.024904
        79      4               5 0.009195
        68     26              41 0.007289
        59      7              12 0.006705
        18     28              55 0.005852
        45      6              13 0.005305
        62     24              56 0.004926
        29     11              26 0.004863

Selected proteins: 30
     Protein  Community  Community size  Katz score  Dubious  Rank  Distance from HSC82  Eligible Rank  Density
4932.YLR378C         64              14    1.698443    False     1                    2              1 0.074713
4932.YBR101C         64              14    1.512181    False     2                    1              2 0.074713
4932.YDR086C         64              14    1.448089    False     3                    2              3 0.074713
4932.YMR203W         85             

In [14]:
top30.to_clipboard(index=False, excel=True)